# Intelligent Calculator

This notebook demonstrates deterministic calculator tools for adding whole-number quantities and calculating horizontal tree distance on level ground. The saved outputs below capture the required agent runs.

> **Credentials:** Provide `GROQ_API_KEY` and, if enabling tracing, `LANGCHAIN_API_KEY` through the hidden prompts. Do not commit credentials.


In [1]:
%pip install -q 'langchain>=1,<2' 'langchain-groq<2' 'langsmith'

In [2]:
import os
from getpass import getpass
from langchain_groq import ChatGroq

if not os.environ.get('GROQ_API_KEY'):
    os.environ['GROQ_API_KEY'] = getpass('Enter your Groq API key: ').strip()
if not os.environ['GROQ_API_KEY']:
    raise ValueError('Enter a Groq API key in the hidden prompt.')
model = ChatGroq(model='openai/gpt-oss-20b', temperature=0, timeout=30, max_retries=1)
print('Model client configured.')

Model client configured.


In [3]:
from math import isfinite, radians, tan
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_core.messages import ToolMessage

@tool
def add(a: int, b: int) -> int:
    """Add two whole-number quantities when they are combined."""
    return a + b

@tool
def tree_distance(height_m: float, angle_deg: float) -> float | str:
    """Find horizontal distance on level ground using height above eye level (metres) and elevation angle (degrees)."""
    if not isfinite(height_m) or height_m <= 0:
        return 'Invalid height. Supply a positive finite height above eye level.'
    if not isfinite(angle_deg) or not 0 < angle_deg < 90:
        return 'Invalid angle. Supply an angle strictly between 0 and 90 degrees.'
    return round(height_m / tan(radians(angle_deg)), 2)

INSTRUCTIONS = (
    'You are the Intelligent Calculator. Use add for combining two whole-number quantities. ' 
    'Use tree_distance only for horizontal distance on level ground when height above eye level in metres ' 
    'and angle of elevation in degrees are supplied. For supported complete problems, call the matching ' 
    'tool once and use its result. If a required value is missing, ask for it before calling a tool. ' 
    'Never guess inputs. If a tool reports an invalid input, ask for a correction. Give final answers with units.'
)
agent = create_agent(model=model, tools=[add, tree_distance], system_prompt=INSTRUCTIONS)

def show_run(problem):
    result = agent.invoke({'messages': [('human', problem)]}, config={'recursion_limit': 8, 'run_name': 'intro_calculator'})
    count = 0
    for message in result['messages']:
        for call in getattr(message, 'tool_calls', []):
            count += 1
            print('Tool requested:', call['name'], call['args'])
        if isinstance(message, ToolMessage):
            print('Tool returned:', message.name, message.content)
    print('Tool requests:', count)
    print('Final answer:', result['messages'][-1].content)
    return result

## Required agent runs
The following cells retain visible output, including tool requests, tool results, tool-call counts, and final answers.

In [4]:
APPLES = 'One person brings five apples and another brings ten. They put them together. How many apples are there?'
apples_run = show_run(APPLES)

Tool requested: add {'a': 5, 'b': 10}
Tool returned: add 15
Tool requests: 1
Final answer: There are **15 apples** in total.


In [5]:
TREE = ('On level ground, the top of a tree is 10 metres above my eye level. '
        'The angle of elevation to the top is 30 degrees. How far away is the tree horizontally?')
tree_run = show_run(TREE)

Tool requested: tree_distance {'angle_deg': 30, 'height_m': 10}
Tool returned: tree_distance 17.32
Tool requests: 1
Final answer: The horizontal distance from you to the tree is **17.32 m**.


In [6]:
MISSING_ANGLE = ('On level ground, the top of a tree is 10 metres above my eye level. '
                 'How far away is the tree horizontally?')
missing_run = show_run(MISSING_ANGLE)

Tool requests: 0
Final answer: I can calculate the horizontal distance using the tree_distance tool, but I need the angle of elevation (in degrees). Could you please provide that?


In [7]:
FRESH = 'One person brings seven apples and another brings eight. How many apples do they have together?'
fresh_run = show_run(FRESH)

Tool requested: add {'a': 7, 'b': 8}
Tool returned: add 15
Tool requests: 1
Final answer: They have **15 apples** together.


## Observability status

The saved outputs above document the calculator runs only. LangSmith tracing and Inspector/MCP access were reported as unavailable/restricted in the original environment; this notebook does not claim those checks were demonstrated, and it includes no fabricated trace IDs, screenshots, or MCP results. To verify tracing, configure a LangSmith key and inspect the resulting run in the LangSmith project. An Inspector/MCP check requires an accessible endpoint and should be documented with actual results when available.
